In [1]:
import os
from tqdm import tqdm
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Conv2DTranspose, Reshape
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Conv2DTranspose, Reshape, LeakyReLU, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

2025-12-08 20:56:11.064978: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 20:56:11.071435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765223771.079169 3290187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765223771.081755 3290187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765223771.087889 3290187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

GPU Memory Growth Enabled


In [3]:
def load_dataset():
    (x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_train = np.expand_dims(x_train, axis=-1)
    x_train = x_train * 2.0 - 1.0
    print("Loaded MNIST dataset, shape:", x_train.shape,
          "min:", x_train.min(), "max:", x_train.max())
    return x_train

BATCH_SIZE = 128
LATENT_DIM = 100
EPOCHS = 100

def display(images, epoch='', name='', n=3, save=False, scale=False):
    if scale:
         images = (images + 1) / 2.0
    for index in range(n * n):
        plt.subplot(n, n, 1 + index)
        plt.axis('off')
        plt.imshow(images[index].squeeze(), cmap='gray')
    fig = plt.gcf()
    fig.suptitle(name + ' ' + str(epoch), fontsize=14)
    if save:
        filename = 'results/generated_plot_e%03d_f.png' % (epoch+1)
        plt.savefig(filename)
        plt.close()
    plt.show()

def grid_plot(images, epoch='', name='', n=3, save=False, scale=False):
    display(images, epoch=epoch, name=name, n=n, save=save, scale=scale)


def build_discriminator(in_shape):
    inputs = tf.keras.Input(shape=in_shape)
    # 5 x5 kernel, as in DCGAN paper it is said it works besyt
    x = Conv2D(64, kernel_size=(5, 5), strides=(2, 2), padding='same',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(inputs)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.1)(x)

    x = Conv2D(128, kernel_size=(5, 5), strides=(2, 2), padding='same',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.1)(x)

    # Output
    x = Flatten()(x)
    # x = Dense(1, activation='sigmoid',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = Dense(1,kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x) # No activation, try afterwards


    model = tf.keras.Model(inputs=inputs, outputs=x, name='Discriminator')
    return model

def build_generator(latent_dim, color=False):
    inputs = tf.keras.Input(shape=(latent_dim,))

    # 7x7 image
    n_nodes = 128 * 7 * 7
    x = Dense(n_nodes,kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(inputs)
    x = Reshape((7, 7, 128))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    # Upsample to 14x14
    x = Conv2DTranspose(128, (5,5), strides=(2,2), padding='same',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    # Upsample to 28x28
    x = Conv2DTranspose(128, (5,5), strides=(2,2), padding='same',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    # Output layer
    dim = 3 if color else 1
    x = Conv2D(dim, (7,7), activation='tanh', padding='same',kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)

    model = tf.keras.Model(inputs=inputs, outputs=x, name='Generator')
    return model
BATCH_SIZE = 64
LATENT_DIM = 100
EPOCHS = 100
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    smooth_real_labels = tf.random.uniform(tf.shape(real_output), 0.8, 1.1)
    # Real loss
    real_loss = cross_entropy(smooth_real_labels, real_output)
    # Fake loss
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):

    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
@tf.function # Compile function for speed
def train_step(images, generator, discriminator):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


def train(kaggle=False, color=False, shape=(28,28,1), name='bw'):
    if kaggle and not color:
        path = '/vol/home/s3328007/.cache/kagglehub/datasets/splcher/animefacedataset/versions/3'
        model_dir = './training_models_kaggle'
        x_train = load_dataset_kaggle(path)
    elif kaggle and color:
        path = '/vol/home/s3328007/.cache/kagglehub/datasets/splcher/animefacedataset/versions/3'
        model_dir = './training_models_kaggle_color'
        x_train = load_dataset_kaggle_color(path)
    else:
        model_dir = './training_models'
        x_train = load_dataset()
    try:
        os.makedirs(model_dir, exist_ok=True)
    except:
        pass
    # Batch and shuffle the data
    train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(BATCH_SIZE)

    generator = build_generator(LATENT_DIM, color=color)
    discriminator = build_discriminator(shape)
    history = {'d_loss': [], 'g_loss': []}
    print("Starting training...")
    try:
        for epoch in range(EPOCHS):
            d_losses = []
            g_losses = []

            for image_batch in tqdm(train_dataset, desc=f"Epoch {epoch+1}/{EPOCHS}"):
                g_loss, d_loss = train_step(image_batch, generator, discriminator)
                d_losses.append(d_loss)
                g_losses.append(g_loss)

            # Calculate average for the epoch
            epoch_d_loss = np.mean(d_losses)
            epoch_g_loss = np.mean(g_losses)


            history['d_loss'].append(epoch_d_loss)
            history['g_loss'].append(epoch_g_loss)

            print(f"Epoch {epoch+1} result: D Loss: {epoch_d_loss:.4f}, G Loss: {epoch_g_loss:.4f}")
            # Save images every 5 epochs
            if (epoch + 1) % 5 == 0:
                save_images(generator, epoch + 1, name)
        generator.save(os.path.join(model_dir, f'generator_final_{name}.h5'))
        discriminator.save(os.path.join(model_dir, f'discriminator_final_{name}.h5'))
        df = pd.DataFrame(history)
        df.to_csv('training_history_5_{name}.csv', index=False)

    except KeyboardInterrupt as e:
        generator.save(os.path.join(model_dir, f'generator_final_{name}.h5'))
        discriminator.save(os.path.join(model_dir, f'discriminator_final_{name}.h5'))
        df = pd.DataFrame(history)
        df.to_csv('training_history_5.csv_{name}', index=False)

    print("Loss history saved to 'training_history.json'")

def save_images(model, epoch,name):
    predictions = model(tf.random.normal([16, LATENT_DIM]), training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.savefig(f'image_at_epoch_{epoch:04d}_{name}_5.png')
    plt.close()
    print(f"Saved image_at_epoch_{epoch:04d}_{name}5.png")


I0000 00:00:1765223781.240056 3290187 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4915 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
if __name__ == "__main__":
    dataset = load_dataset()
    train()

Loaded MNIST dataset, shape: (60000, 28, 28, 1) min: -1.0 max: 1.0


/vol/home/s3328007/.conda/envs/idl2/lib/python3.10/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Loaded MNIST dataset, shape: (60000, 28, 28, 1) min: -1.0 max: 1.0
Starting training...


Epoch 1/100: 100%|██████████| 938/938 [00:18<00:00, 51.26it/s]


Epoch 1 result: D Loss: 1.2806, G Loss: 0.8395


Epoch 2/100: 100%|██████████| 938/938 [00:16<00:00, 55.78it/s]


Epoch 2 result: D Loss: 1.2329, G Loss: 0.9209


Epoch 3/100:  64%|██████▍   | 605/938 [00:10<00:05, 55.98it/s]

In [ ]:
#generate 100 images from trained generator
def generate_images(model, num_images=100):
    noise = tf.random.normal([num_images, LATENT_DIM])
    generated_images = model(noise, training=False)
    generated_images = (generated_images + 1) / 2.0  # Rescale to [0, 1]
    fig = plt.figure(figsize=(10, 10))
    for i in range(100):
        plt.subplot(10, 10, i + 1)
        plt.imshow(generated_images[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.show()
    # return generated_images
model = tf.keras.models.load_model('training_models/generator_final.h5')
generate_images(model)

2025-12-08 16:52:31.409222: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 10.99MiB (rounded to 11520000)requested by op Conv2DBackpropInput
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-12-08 16:52:31.409276: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-12-08 16:52:31.409300: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 13, Chunks in use: 13. 3.2KiB allocated for chunks. 3.2KiB in use in bin. 64B client-requested in use in bin.
2025-12-08 16:52:31.409310: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 14, Chunks in use: 14. 7.0KiB allocated for chunks. 7.0KiB in use in bin. 7.0KiB client-requested in use in bin.
202

ResourceExhaustedError: Exception encountered when calling Conv2DTranspose.call().

[1m{{function_node __wrapped__Conv2DBackpropInput_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[100,15,15,128] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator GPU_0_bfc [Op:Conv2DBackpropInput][0m

Arguments received by Conv2DTranspose.call():
  • inputs=tf.Tensor(shape=(100, 7, 7, 128), dtype=float32)

In [4]:
import kagglehub
from PIL import Image
# Download latest version
path = kagglehub.dataset_download("splcher/animefacedataset")


def load_dataset_kaggle(data_path, image_size=(28, 28)):
    images_list = []
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    data_path = os.path.join(data_path, 'images')
    i =0

    for filename in os.listdir(data_path):
            i+=1
            if i % 1000 == 0:
                print(i)
            ext = os.path.splitext(filename)[1].lower()
            if ext not in valid_extensions:
                continue

            img_path = os.path.join(data_path, filename)

            try:
                img = Image.open(img_path).convert('L')
                img = img.resize(image_size)
                images_list.append(np.array(img))
            except:
                continue
            # if i > 10000:
            #      break


    x_train = np.array(images_list)
    x_train = x_train.astype('float32') / 255.0
    x_train = np.expand_dims(x_train, axis=-1)
    x_train = x_train * 2.0 - 1.0
    print("Loaded dataset, shape:", x_train.shape, "min:", x_train.min(), "max:", x_train.max())

    return x_train


/vol/home/s3328007/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train(kaggle=True, shape=(68,68,1), name='kaggle_bw')

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000


In [ ]:
def load_dataset_kaggle_color(data_path, image_size=(28, 28)):


    images_list = []
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

    if not os.path.exists(data_path):
        print(f"Path not found: {data_path}")
        return None



    for filename in os.listdir(data_path):
            i+=1
            if i % 1000 == 0:
                print(i)
            ext = os.path.splitext(filename)[1].lower()
            if ext not in valid_extensions:
                continue

            img_path = os.path.join(data_path, filename)

            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize(image_size)
                images_list.append(np.array(img))
            except:
                continue
            # if i > 10000:
            #      break

    if len(images_list) == 0:
        print('No images found')
        return np.array([])

    x_train = np.array(images_list)
    # Normalization stays the same [-1, 1]
    x_train = x_train.astype('float32') / 127.5 - 1.0
    print(f"Success! Loaded {len(x_train)} COLOR images. Shape: {x_train.shape}")
    # Shape should now be roughly (10000, 28, 28, 3)
    return x_train

In [ ]:
train(kaggle=True, color=True, shape=(68,68,3), name='kaggle_color')

Failed to load 0_2000.jpg: name 'Image' is not defined
Failed to load 10000_2004.jpg: name 'Image' is not defined
Failed to load 10001_2004.jpg: name 'Image' is not defined
Failed to load 10002_2004.jpg: name 'Image' is not defined
Failed to load 10003_2004.jpg: name 'Image' is not defined
Failed to load 10004_2004.jpg: name 'Image' is not defined
Failed to load 10005_2004.jpg: name 'Image' is not defined
Failed to load 10006_2004.jpg: name 'Image' is not defined
Failed to load 10007_2004.jpg: name 'Image' is not defined
Failed to load 10008_2004.jpg: name 'Image' is not defined
Failed to load 10009_2004.jpg: name 'Image' is not defined
Failed to load 1000_2000.jpg: name 'Image' is not defined
Failed to load 10010_2004.jpg: name 'Image' is not defined
Failed to load 10011_2004.jpg: name 'Image' is not defined
Failed to load 10012_2004.jpg: name 'Image' is not defined
Failed to load 10013_2004.jpg: name 'Image' is not defined
Failed to load 10014_2004.jpg: name 'Image' is not defined
Fa

/vol/home/s3328007/.conda/envs/idl2/lib/python3.10/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


ValueError: Input 0 of layer "conv2d_1" is incompatible with the layer: expected min_ndim=4, found ndim=1. Full shape received: (None,)